# Organizational hierarchies

Two teams participate in two business lines. This example compares both orders on the same temporary payroll rows and reconciles a team's organizational contribution to its local payroll change.

In [ ]:
import pandas as pd

from labor_cost import (
    AnalysisRequest,
    DatasetSnapshot,
    analyze,
    labor_cost_model,
)

rows = []
for month, counts, rates in [
    ("2025-01-01", (10, 30, 30, 30), (10, 10, -1, 3)),
    ("2025-02-01", (10, 30, 70, 90), (13, 13, 1, 4)),
]:
    for cell, (count, rate) in enumerate(zip(counts, rates)):
        for worker in range(count):
            rows.append((month, f"{cell}-{worker}", "ГПД", "AB"[cell // 2],
                         "XY"[cell % 2], rate, rate))
source = pd.DataFrame(rows, columns=[
    "period", "mdm_employee_rk_hash", "motivation_type",
    "lvl7_mapped_management_unit_nm", "business_line", "gpd_pay", "actual"
])
dataset = DatasetSnapshot("worked-example", "1", source)
model = labor_cost_model(schema_version=2, line_source="business_line")

In [ ]:
runs = {}
for order in [("team", "line"), ("line", "team")]:
    request = AnalysisRequest("2025-01", "2025-02", hierarchy=order,
                              source_cost_column="actual")
    runs[order] = analyze(model, dataset, request)
pd.concat({str(order): run.table("driver_bridge").set_index("driver").impact
           for order, run in runs.items()}, axis=1)

## Team A in the team-first analysis

Its exposure stays at 40 and unit cost rises from 10 to 13. The local bridge explains 120. In the full organization, exposure doubles and the team's share halves. The outer scale/share game assigns 130 to the team's economics.

In [ ]:
run = runs["team", "line"]
team = run.select_node((("team", "A"),))
accounting = pd.Series({
    "Observed payroll change": team["node_summary"].cost_change.iloc[0],
    "Local driver impacts": team["local_bridge"].impact.sum(),
    "Organizational economics contribution":
        team["organization_contributions"].impact.sum(),
})
assert abs(accounting.iloc[0] - 120) < 1e-6
assert abs(accounting.iloc[1] - 120) < 1e-6
assert abs(accounting.iloc[2] - 130) < 1e-6
accounting

In [ ]:
pair = run.select_node((("team", "A"), ("line", "Y")))
pair["local_bridge"][["label", "impact", "period_A_cost", "period_B_cost"]]

Set `hierarchy=("line",)` to pool payroll by line, or `("team",)` to pool by team. Inspect `node_summary` for the available node paths. `report(run, nodes=[path])` displays a selected local bridge and its organizational contributions.